In [246]:
marc_rda_relators = {
    '''Creates dictionary of relator codes common to MARC and RDA'''
    "ape": "appellee",
    "apl": "appellant",
    "arc": "architect",
    "art": "artist",
    "aup": "audio producer",
    "aus": "screenwriter",
    "aut": "author",
    "bka": "book artist",
    "cad": "casting director",
    "chr": "choreographer",
    "cll": "calligrapher",
    "cmp": "composer",
    "com": "compiler",
    "cou": "court governed",
    "csl": "consultant",
    "ctg": "cartographer",
    "dfd": "defendant",
    "dgc": "degree committee member",
    "dgg": "degree granting institution",
    "dgs": "degree supervisor",
    "drt": "director",
    "dsr": "designer",
    "dte": "dedicatee",
    "dto": "dedicator",
    "edd": "editorial director",
    "enj": "enacting jurisdiction",
    "fmd": "film director",
    "fmk": "filmmaker",
    "fmp": "film producer",
    "his": "host institution",
    "inv": "inventor",
    "isb": "issuing body",
    "ive": "interviewee",
    "ivr": "interviewer",
    "jud": "judge",
    "jug": "jurisdiction governed",
    "lbt": "librettist",
    "lsa": "landscape architect",
    "lyr": "lyricist",
    "med": "medium",
    "orm": "organizer",
    "pht": "photographer",
    "pra": "praeses",
    "prg": "programmer",
    "prn": "production company",
    "pro": "producer",
    "ptf": "plaintiff",
    "rap": "rapporteur",
    "rcp": "addressee",
    "rdd": "radio director",
    "res": "researcher",
    "rpc": "radio producer",
    "rsp": "respondent",
    "rxa": "remix artist",
    "scl": "sculptor",
    "tld": "television director",
    "tlp": "television producer",
}

In [251]:
import re, collections, requests
from lxml import etree

'''Gets xml file and sets tree and root'''
tree = etree.parse(r"C:\Users\yello\anaconda3\Cook\MC122_20260122_190101_UTC__ead.xml")
root = tree.getroot()

'''Strips namespace from xml file (This portion of code was fully generated by ChatGPT 5.2)'''
for elem in root.getiterator():
    if not hasattr(elem.tag, 'find'):
        continue
    i = elem.tag.find('}')
    if i >= 0:
        elem.tag = elem.tag[i+1:]

'''Get target <c0*> tag (This portion of code was partially revised utilizing ChatGPT 5.2)'''
target_id = input("Enter aspace id of target <c0*>: ").strip()

result = root.xpath(
    "//*[starts-with(local-name(), 'c0') and @id=$t_id]",
    t_id=target_id
)

'''Set global variables'''
c0_raw = result[0]
c0_print = etree.tostring(c0_raw, pretty_print=True, encoding="unicode")

'''Set global names lists'''
names_list = c0_raw.xpath(".//*[local-name()='origination']") #Troubleshot using ChatGPT 5.2

all_persnames_list = []
all_corpnames_list = []

creator_names_list = []
creator_persnames_list = []
creator_corpnames_list = []

subject_names_list = []
subject_persnames_list = []
subject_corpnames_list = []

source_names_list = []
source_persnames_list = []
source_corpnames_list = []

for orig in names_list:
    if orig.attrib["label"].lower() == "creator":
        creator_names_list.append(orig)
    elif orig.attrib["label"].lower() == "subject":
        subject_names_list.append(orig)
    elif orig.attrib["label"].lower() == "source":
        source_names_list.append(orig)

for orig in names_list:
    if orig.xpath(".//*[local-name()='persname']"):
        all_persnames_list.append(orig)
        if orig.attrib["label"].lower() == "creator":
            creator_persnames_list.append(orig)
        elif orig.attrib["label"].lower() == "subject":
            subject_persnames_list.append(orig)
        elif orig.attrib["label"].lower() == "source":
            source_persnames_list.append(orig)
    if orig.xpath(".//*[local-name()='corpname']"):
        all_corpnames_list.append(orig)
        if orig.attrib["label"].lower() == "creator":
            creator_corpnames_list.append(orig)
        elif orig.attrib["label"].lower() == "subject":
            subject_corpnames_list.append(orig)
        elif orig.attrib["label"].lower() == "source":
            source_corpnames_list.append(orig)

### TO DO: ADD SUPPORT FOR OTHER ASPACE AGENT TYPES (FAMILY, SOFTWARE)

# OLD NAMES LISTS:
# names_list = c0_raw.xpath(".//*[local-name()='origination']//*[contains(local-name(), 'name')]") #Troubleshot using ChatGPT 5.2
# persnames_list = c0_raw.xpath(".//origination//persname")
# corpnames_list = c0_raw.xpath(".//origination//corpname")

print (c0_print)

Enter aspace id of target <c0*>:  aspace_e32cb1c4f4db3b1932b5ca9a945901aa


<c03 xmlns="urn:isbn:1-931666-22-9" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" id="aspace_e32cb1c4f4db3b1932b5ca9a945901aa" level="item">
  <did>
    <unittitle>סויטה קטנה לחליל גדול = A little suite for a big flute (flauto basso in do)</unittitle>
    <origination audience="internal" label="Creator">
      <persname authfilenumber="120307688" role="cmp" rules="rda" source="viaf">Steinberg, Zeev, 1918-2011</persname>
    </origination>
    <physdesc altrender="whole">
      <extent altrender="materialtype spaceoccupied">1 folder(s)</extent>
      <extent altrender="carrier">1 score (5 pages)</extent>
      <dimensions>35 cm</dimensions>
    </physdesc>
    <unitdate datechar="creation" normal="1972/1972">1972</unitdate>
    <unitdate datechar="copyright" normal="1973/1973">c1973</unitdate>
    <langmaterial><language langcode="heb" scriptcode="Hebr">Hebrew</language>, <language langcode="eng" scriptcode="Latn">English</language>.</l

In [32]:
def ead2marc_245():

    '''INDICATORS'''
    '''Indicator 1 is constant (1)'''
    '''Indicator 2 is constant (0)'''

    
    '''SUBFIELD A'''
    '''Get clean title (This portion of code was partially revised utilizing ChatGPT 5.2)'''
    title_fetch = c0_raw.xpath(".//*[local-name()='unittitle']")
    title_raw = title_fetch[0]
    title_clean = title_raw.xpath("string()").strip()
    a_245 = f"""<marc:subfield code="a">{title_clean}</marc:subfield>"""
    
    '''PRINT 245 FIELD'''
    print("""<marc:datafield tag="245" ind1="1" ind2="0">""")
    print(a_245)
    print("""</marc:datafield""")

ead2marc_245()

<marc:datafield tag="245" ind1="1" ind2="0">
<marc:subfield code="a">Canonic pieces for stringquartet or stringorchestra</marc:subfield>
</marc:datafield


In [61]:
def ead2marc_264():
    '''Create a list of all unitdates'''
    unitdates_list = c0_raw.xpath(".//*[local-name()='unitdate']")

    
    '''Complete INDICATORS through PRINT 264 Field for each unitdate'''
    for unitdate in unitdates_list:

        '''INDICATORS'''
        '''Indicator 1 is constant (blank)'''
    
        '''Indicator 2 (This portion of code was partially revised utilizing ChatGPT 5.2)'''
        '''Get date label (datechar)'''
        date_clean = unitdate.xpath("string()").strip()
        datechar_clean = (unitdate.get("datechar")).strip()
        
        '''Set indicator 2 based on datechar'''
        if datechar_clean in ['broadcast', 'publication']:
            ind2_264 = 1
        elif datechar_clean in ['issued']:
            ind2_264 = 3
        elif datechar_clean in ['copyright']:
            ind2_264 = 4
        else: 
            #AKA elif datechar_clean in ['creation', 'deaccession', 'digitized', 'event', 'existence', 'modified', 'other', 'record keeping', 'usage']
            ind2_264 = 0
    
        
        '''SUBFIELD C'''
        '''Set date to read [not after dddd] if datechar is "deaccession" (Laikin's idea)'''
        if datechar_clean in ['deaccession']:
            qualifier_open = "[not after "
            qualifier_close = "]"
        else:
            qualifier_open = ""
            qualifier_close = ""
    
        c_264 = f"""<marc:subfield code ="c">{qualifier_open}{date_clean}{qualifier_close}</marc:subfield>"""
    
        
        '''PRINT 264 FIELD'''
        print(f"""<marc:datafield tag="264" ind1=" " ind2="{ind2_264}">""")
        print (c_264)
        print("""</marc:datafield""")
        print("")

ead2marc_264()

<marc:datafield tag="264" ind1=" " ind2="0">
<marc:subfield code ="c">[before 2011?]</marc:subfield>
</marc:datafield



In [252]:
def ead2marc_100_110():

    
    '''Set function-wide variables'''
    main_name_ead = creator_names_list[0]
    main_onlyname_ead_list = main_name_ead.xpath(".//*[contains(local-name(), 'name')]")
    main_onlyname_ead = main_onlyname_ead_list[0]
    
    printme = etree.tostring(main_name_ead, pretty_print=True, encoding="unicode")
    print(printme)


    '''Determine whether first name is persname or corpname to determine if field is 100 or 110'''
    if main_name_ead in creator_persnames_list:
        field_100_or_110 = "100"
    elif main_name_ead in creator_corpnames_list:
        field_100_or_110 = "110"
    
    ### TO DO: CREATE SOME WAY FOR USER TO TOGGLE WHICH NAME THEY WANT TO BE 100/110 INSTEAD OF JUST SETTING IT TO FIRST LISTED CREATOR.
    
    ### TO DO: CREATE CASES FOR OTHER EAD NAME ELEMENTS (agencyname, famname, geogname, namegrp)
        
    '''Check if main name is associated with an authority file (if statement partially revised utilizing ChatGPT 5.2)'''
    if main_onlyname_ead_list and main_onlyname_ead_list[0].get("source") in {"lcnaf", "naf"}:
        authority = "lcnaf"
        authfile_no = main_onlyname_ead.get("authfilenumber")
        authority_url = f"""https://lccn.loc.gov/{authfile_no}/marcxml"""
        authority_xml = requests.get(authority_url).content
        authority_root = etree.fromstring(authority_xml)
        authority_100_110_list = authority_root.xpath(".//*[local-name()='datafield' and @tag='100']")
        authority_100_110 = authority_100_110_list[0]
        authority_subfields = authority_100_110.xpath(".//*[local-name()='subfield']")
        '''Create dictionary of subfields (This portion of code was fully genereated by ChatGPT 5.2)'''
        subfield_dict = {}
        for subfield in authority_subfields:
            code = subfield.get("code")
            value = subfield.text.strip() if subfield.text else ""
            subfield_dict[code] = value
        print(etree.tostring(authority_100_110, pretty_print=True, encoding="unicode"))
    elif main_onlyname_ead_list and main_onlyname_ead_list[0].get("source") == "viaf":
        authority = "viaf"
        # TO DO: FIGHT THE VIAF GODS (figue out how to get MARCXML from VIAF)
    else:
        authority = None  
    ##TO DO: ADD OTHER AUTHORITIES (EX. VIAF)
        
    
    '''INDICATORS'''
    '''Indicator 1 (This portion of code was partially revised utilizing ChatGPT 5.2)'''
    if authority == "lcnaf":
        ind1_100_110 = authority_100_110.attrib["ind1"]
    # elif authority == "viaf":
        
    else:
        print("nope!")

    
    '''Indicator 2 is constant (blank)'''


    '''SUBFIELD A'''
    if authority == "lcnaf":
        a_content = subfield_dict["a"]
    # elif authority == "viaf":
    #     TO DO: FIGHT THE VIAF GODS
    else:
        a_content = main_name_ead.xpath("string()").strip()
    ### TO DO: SEPARATE ELSE CASE MAIN_NAME_CLEAN CONTENTS INTO SUBFIELDS D AND E
    a_100_110 = f"""<marc:subfield code ="a">{a_content}</marc:subfield>"""


    '''SUBFIELD C'''
    c_content = None
    if authority == "lcnaf":
        c_test = subfield_dict.get("c")
        if c_test is not None:
            c_content = subfield_dict["c"]
    # elif authority == "viaf":
    #     TO DO: FIGHT THE VIAF GODS
    else:
        c_content = None
    if c_content is not None:
        c_100_110 = f"""<marc:subfield code ="c">{c_content}</marc:subfield>"""
    else:
        c_100_110 = None
    
    
    '''SUBFIELD D'''
    if authority == "lcnaf":
        d_content = subfield_dict["d"]
    # elif authority == "viaf":
    #     TO DO: FIGHT THE VIAF GODS
    else:
        d_content = 

    
    d_100_110 = f"""<marc:subfield code ="d">{d_content}</marc:subfield>"""

    
    '''SUBFIELD E'''
    aspace_relator = main_onlyname_ead.attrib["role"].lower()
    if aspace_relator in marc_rda_relators.keys():
        e_content = marc_rda_relators[aspace_relator]
    else:
        e_contnet = None
    if e_content is not None:
        e_100_110 = f"""<marc:subfield code ="e">{e_content}</marc:subfield>"""
    else:
        c_100_110 = None


    '''PRINT 100/110 FIELD'''
    print(f"""<marc:datafield tag="{field_100_or_110}" ind1="{ind1_100_110}" ind2=" ">""")
    print(a_100_110)
    if c_100_110:
        print(c_100_110)
    print(d_100_110)
    if e_100_110:
        print(e_100_110)
    print("""</marc:datafield>""")
    print("")


ead2marc_100_110()

<origination xmlns="urn:isbn:1-931666-22-9" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" audience="internal" label="Creator">
  <persname authfilenumber="120307688" role="cmp" rules="rda" source="viaf">Steinberg, Zeev, 1918-2011</persname>
</origination>

nope!


UnboundLocalError: cannot access local variable 'd_content' where it is not associated with a value

In [ ]:
def ead2marc_700():
    '''Create a list of all names and a list of all persnames'''

    '''Complete INDICATORS through PRINT 700 Field for each persname'''
    for persname in persname_list:
        
        '''INDICATORS'''
        '''Indicator 1'''
        
    
        '''Indicator 2 is constant (blank)'''
    


ead2marc_100()